In [1]:
import pandas as pd
import numpy as np

### Reading in master hospital table:

In [23]:
hospitals_master = pd.read_csv('../data/hospitals_master.csv')

In [24]:
hospitals_master.shape

(853, 15)

In [25]:
hospitals_master.columns

Index(['CCN', 'Facility Name', 'Emergency Services', 'Hospital Ownership',
       'Closed', 'Closure Date', 'Full Address', 'Prior Name', 'RUCA',
       'Tract_Code', 'CBSA_Code', 'CBSA_Title', 'Metro_Status', 'County_FIPS',
       'State_FIPS'],
      dtype='object')

In [27]:
hospitals_master['Closed'].value_counts()

Closed
0    815
1     20
2     18
Name: count, dtype: int64

I'll need to filter out any hospitals that closed before 2010, since the financial data I'll be using only goes back to 2010. This will affect 5 hospitals (13% of the closed or converted hospitals in the dataset).

In [45]:
hospitals_master['Closure Date'] = pd.to_datetime(hospitals_master['Closure Date'])

In [34]:
hospitals_master[~hospitals_master['Closure Date'].isna()]['Closure Date'].sort_values()

367   2005-03-01
511   2005-10-01
191   2005-12-01
559   2007-11-01
583   2009-11-01
249   2011-03-01
581   2012-03-01
212   2014-10-01
540   2015-08-01
708   2016-01-01
485   2020-01-01
658   2020-08-01
533   2020-10-01
678   2020-10-01
179   2021-06-01
464   2021-07-01
253   2022-01-01
2     2022-02-01
92    2022-09-01
32    2022-12-01
784   2023-03-01
706   2023-06-01
416   2023-08-01
339   2023-10-01
312   2023-10-01
682   2023-10-01
707   2023-10-01
355   2024-03-01
36    2024-04-01
182   2024-05-01
465   2024-09-01
795   2025-01-01
529   2025-05-01
269   2025-09-01
161   2025-10-01
100   2026-05-01
Name: Closure Date, dtype: datetime64[ns]

In [49]:
hospitals_master = hospitals_master[~(hospitals_master['Closure Date'] < '2010-01-01')]

In [50]:
hospitals_master.head()

,CCN,Facility Name,Emergency Services,Hospital Ownership,Closed,Closure Date,Full Address,Prior Name,RUCA,Tract_Code,CBSA_Code,CBSA_Title,Metro_Status,County_FIPS,State_FIPS
0,190044,ACADIA GENERAL HOSPITAL,Yes,Voluntary non-profit - Private,0,NaT,"1305 CROWLEY RAYNE HIGHWAY, CROWLEY, LA, 70526",NaN,4.0,960801.0,29180.0,"Lafayette, LA",Metropolitan Statistical Area,22001,22.0
1,390163,ACMH HOSPITAL,Yes,Voluntary non-profit - Private,0,NaT,"1 NOLTE DRIVE, KITTANNING, PA, 16201",NaN,4.0,950500.0,38300.0,"Pittsburgh, PA",Metropolitan Statistical Area,42005,42.0
2,320070,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,Yes,"Governmental, Federal",1,2022-02-01,"80B VETERANS BLVD, ACOMA, NM, 87034",NaN,2.0,941500.0,NaN,NaN,Neither,35006,35.0
3,360159,ADENA REGIONAL MEDICAL CENTER,Yes,Voluntary non-profit - Private,0,NaT,"272 HOSPITAL ROAD, CHILLICOTHE, OH, 45601",NaN,4.0,956300.0,17060.0,"Chillicothe, OH",Micropolitan Statistical Area,39141,39.0
4,330079,ADIRONDACK MEDICAL CENTER - SARANAC LAKE,Yes,Voluntary non-profit - Other,0,NaT,"2233 STATE ROUTE 86, SARANAC LAKE, NY, 12983",NaN,7.0,950900.0,NaN,NaN,Neither,36033,36.0


### Gathering hospital financial data:

Hospital financial data aggregation was performed in the "Financial Report Data Gathering Notebook" notebook with the help of code drawn from the HCRIS-databuilder GitHub project (https://github.com/Rush-Quality-Analytics/HCRIS-databuilder/tree/master).

In [39]:
financials = pd.read_csv('../data/HCRIS_databuilder/filtered_datasets/HCRIS_filtered.csv').drop(columns=['Fiscal Year Begin Date','Fiscal Year End Date']).rename(columns={'Beginning FFY':'Year',': General Ownership Type':'General Ownership Type'})

                                                                                                         

In [40]:
financials.columns

Index(['Hospital', 'Hospital type', 'Control type',
       'REIMBURSEMENT SETTLEMENT: Subtotal',
       'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)',
       'BALANCE SHEET: Inventory (G_C1THRU4_7)',
       'NUMBER OF BEDS: Adults & Pediatrics', 'BED DAYS: Total Hospital',
       'IPPS Interim payment',
       'BALANCE SHEET: Temporary Investments (G_C1THRU4_2)',
       'STATEMENT OF REVENUES AND EXPENSES: Net Patient Revenue (G3_C1_3)',
       'Urban (1) and Rural (2) Indicator', 'Cost of Uncompensated Care',
       'RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)',
       'Total Bad Debt expense', 'HAC reduction adjustment amount',
       'Total cost of charity care',
       'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)',
       'BALANCE SHEET: Total Current Liabilities (G_C1THRU4_45)',
       'Total Days Title XVIII', 'Total Costs',
       'ADJUSTED SALARIES, Subtotal Salaries', 'Number of Interns & Residents',
       'Medicaid charges', '

In [43]:
financials[['Facility Name', 'CCN']] = financials['Hospital'].str.extract(r'(.*)\s+\((\d+)\)')
financials = financials.drop(columns='Hospital')

In [44]:
financials.head()

,Hospital type,Control type,REIMBURSEMENT SETTLEMENT: Subtotal,BALANCE SHEET: Total Current Assets (G_C1THRU4_11),BALANCE SHEET: Inventory (G_C1THRU4_7),NUMBER OF BEDS: Adults & Pediatrics,BED DAYS: Total Hospital,IPPS Interim payment,BALANCE SHEET: Temporary Investments (G_C1THRU4_2),STATEMENT OF REVENUES AND EXPENSES: Net Patient Revenue (G3_C1_3),...,Financial Indicators: SOLVENCY Debt-to-Equity Ratio,Financial Indicators: SOLVENCY Debt Ratio,Financial Indicators: SOLVENCY Equity Ratio,Financial Indicators: SOLVENCY Interest Coverage Ratio,Financial Indicators: SOLVENCY total assets less total liabilities,Financial Indicators: EFFICIENCY asset turnover ratio,Financial Indicators: EFFICIENCY Accounts Receivable Turnover Ratio,General Ownership Type,Facility Name,CCN
0,General Short Term (Acute Care Hospitals),Voluntary Nonprofit-Other,1439065.0,10599555.0,1323522.0,136.0,17080.0,2681052.0,NaN,9075929.0,...,1.121873,0.528718,0.471282,NaN,5374708.0,0.795823,1.255900,non-profit,ACADIA GENERAL HOSPITAL,190044
1,General Short Term (Acute Care Hospitals),Voluntary Nonprofit-Other,4186735.0,8985534.0,1400773.0,136.0,51100.0,7623187.0,NaN,35835326.0,...,2.131645,0.680679,0.319321,NaN,3154921.0,3.627023,5.701668,non-profit,ACADIA GENERAL HOSPITAL,190044
2,General Short Term (Acute Care Hospitals),Voluntary Nonprofit-Other,5074148.0,7406753.0,1543664.0,133.0,50142.0,9158722.0,NaN,30808481.0,...,0.836340,0.455439,0.544561,NaN,4550941.0,3.686514,7.387468,non-profit,ACADIA GENERAL HOSPITAL,190044
3,General Short Term (Acute Care Hospitals),Voluntary Nonprofit-Other,6448922.0,10201314.0,1481240.0,175.0,65335.0,10627039.0,NaN,38384736.0,...,0.861173,0.462704,0.537296,NaN,6186036.0,3.333952,7.258461,non-profit,ACADIA GENERAL HOSPITAL,190044
4,General Short Term (Acute Care Hospitals),Voluntary Nonprofit-Other,6878089.0,9280202.0,1532531.0,136.0,51100.0,8623459.0,NaN,36929317.0,...,1.481635,0.597040,0.402960,-10.034497,4603240.0,3.232733,6.751346,non-profit,ACADIA GENERAL HOSPITAL,190044


### Gathering hospital quality data:

### Gathering demographic data: